# Phase 1 — Understand the source

Downloads **one real hour** of the GH Archive event stream and measures it, instead of assuming the spec's estimates hold. Answers:

1. Compressed size, uncompressed size, event count.
2. Distribution of `type`.
3. Full schema tree, and how `payload` varies by event type.
4. Which fields are consistently present vs sparse.
5. Raw JSON vs Parquet (all columns) vs Parquet (projected columns) — real compression ratio.
6. Extrapolated daily/monthly volume, checked against the measured R2 free tier.

In [1]:
import gzip
import json
from collections import Counter
from pathlib import Path

import duckdb
import requests

RAW_DIR = Path("../data/raw_sample")
RAW_DIR.mkdir(parents=True, exist_ok=True)

# WHY this hour: a recent, ordinary weekday (Thursday), a few days in the
# past so GH Archive has definitely finished publishing it. Picked by
# checking the current date, not hardcoded from the spec's example.
HOUR_URL = "https://data.gharchive.org/2026-09-17-15.json.gz"
GZ_PATH = RAW_DIR / "2026-09-17-15.json.gz"
JSON_PATH = RAW_DIR / "2026-09-17-15.json"


## 1. Compressed size, uncompressed size, event count

In [2]:
if not GZ_PATH.exists():
    resp = requests.get(HOUR_URL, stream=True, timeout=60)
    resp.raise_for_status()
    with open(GZ_PATH, "wb") as f:
        for chunk in resp.iter_content(chunk_size=1 << 20):
            f.write(chunk)

compressed_bytes = GZ_PATH.stat().st_size
print(f"Compressed (.json.gz) size: {compressed_bytes:,} bytes "
      f"({compressed_bytes / 1024**2:.2f} MiB)")


Compressed (.json.gz) size: 23,594,269 bytes (22.50 MiB)


In [3]:
with gzip.open(GZ_PATH, "rb") as gz_in, open(JSON_PATH, "wb") as out:
    out.write(gz_in.read())

uncompressed_bytes = JSON_PATH.stat().st_size
with open(JSON_PATH, encoding="utf-8") as f:
    lines = f.readlines()

event_count = len(lines)
print(f"Uncompressed (.json) size: {uncompressed_bytes:,} bytes "
      f"({uncompressed_bytes / 1024**2:.2f} MiB)")
print(f"Event count: {event_count:,}")
print(f"gzip compression ratio: {uncompressed_bytes / compressed_bytes:.2f}x")


Uncompressed (.json) size: 107,955,076 bytes (102.95 MiB)
Event count: 61,147
gzip compression ratio: 4.58x


## 2. Distribution of `type`

In [4]:
events = [json.loads(line) for line in lines]
type_counts = Counter(e.get("type") for e in events)

print("Event type distribution:")
for event_type, count in type_counts.most_common():
    pct = 100 * count / event_count
    print(f"  {event_type:25s} {count:7,d}  ({pct:5.2f}%)")


Event type distribution:
  PushEvent                  40,632  (66.45%)
  PullRequestEvent            5,500  ( 8.99%)
  CreateEvent                 4,031  ( 6.59%)
  IssueCommentEvent           2,841  ( 4.65%)
  IssuesEvent                 2,022  ( 3.31%)
  PullRequestReviewEvent      1,522  ( 2.49%)
  DeleteEvent                 1,324  ( 2.17%)
  PullRequestReviewCommentEvent   1,316  ( 2.15%)
  WatchEvent                  1,056  ( 1.73%)
  ReleaseEvent                  400  ( 0.65%)
  ForkEvent                     278  ( 0.45%)
  MemberEvent                   109  ( 0.18%)
  CommitCommentEvent             64  ( 0.10%)
  PublicEvent                    27  ( 0.04%)
  GollumEvent                    17  ( 0.03%)
  DiscussionEvent                 8  ( 0.01%)


## 3. Schema tree — how deep does `payload` nest, and how does it vary by event type?

In [5]:
def describe_shape(obj, prefix="", max_depth=6, _depth=0):
    """Yield 'dotted.path: type' lines for a nested dict/list, capped at max_depth."""
    if _depth > max_depth:
        yield f"{prefix}: <max depth reached>"
        return
    if isinstance(obj, dict):
        for key, value in obj.items():
            path = f"{prefix}.{key}" if prefix else key
            yield from describe_shape(value, path, max_depth, _depth + 1)
    elif isinstance(obj, list):
        if obj:
            yield from describe_shape(obj[0], f"{prefix}[]", max_depth, _depth + 1)
        else:
            yield f"{prefix}: [] (empty list)"
    else:
        yield f"{prefix}: {type(obj).__name__}"

sample_by_type = {}
for e in events:
    t = e.get("type")
    if t not in sample_by_type:
        sample_by_type[t] = e

print("Schema tree for one sample event per type (top 5 types by volume):\n")
for event_type, _ in type_counts.most_common(5):
    print(f"=== {event_type} ===")
    for line in describe_shape(sample_by_type[event_type]):
        print(f"  {line}")
    print()


Schema tree for one sample event per type (top 5 types by volume):

=== PushEvent ===
  id: str
  type: str
  actor.id: int
  actor.login: str
  actor.display_login: str
  actor.gravatar_id: str
  actor.url: str
  actor.avatar_url: str
  repo.id: int
  repo.name: str
  repo.url: str
  payload.repository_id: int
  payload.push_id: int
  payload.ref: str
  payload.head: str
  payload.before: str
  public: bool
  created_at: str
  org.id: int
  org.login: str
  org.gravatar_id: str
  org.url: str
  org.avatar_url: str

=== PullRequestEvent ===
  id: str
  type: str
  actor.id: int
  actor.login: str
  actor.display_login: str
  actor.gravatar_id: str
  actor.url: str
  actor.avatar_url: str
  repo.id: int
  repo.name: str
  repo.url: str
  payload.action: str
  payload.number: int
  payload.pull_request.url: str
  payload.pull_request.id: int
  payload.pull_request.number: int
  payload.pull_request.head.ref: str
  payload.pull_request.head.sha: str
  payload.pull_request.head.repo.id: in

## 4. Field presence — consistently present vs sparse

In [6]:
def get_path(d, path):
    cur = d
    for part in path.split("."):
        if not isinstance(cur, dict):
            return None
        cur = cur.get(part)
        if cur is None:
            return None
    return cur

# WHY these fields: this is exactly the bronze schema Phase 2 is about to build
# against, so this measurement doubles as validation of that schema.
candidate_fields = [
    "id", "type", "created_at",
    "actor.id", "actor.login",
    "repo.id", "repo.name",
    "org.id",
    "payload.action",
    "payload.pull_request.number", "payload.pull_request.merged",
    "payload.issue.number", "payload.commits",
]

print(f"Field presence across all {event_count:,} events:\n")
for path in candidate_fields:
    present = sum(1 for e in events if get_path(e, path) is not None)
    pct = 100 * present / event_count
    print(f"  {path:32s} {present:7,d} / {event_count:,}  ({pct:5.1f}%)")


Field presence across all 61,147 events:

  id                                61,147 / 61,147  (100.0%)
  type                              61,147 / 61,147  (100.0%)
  created_at                        61,147 / 61,147  (100.0%)
  actor.id                          61,147 / 61,147  (100.0%)
  actor.login                       61,147 / 61,147  (100.0%)


  repo.id                           61,145 / 61,147  (100.0%)


  repo.name                         61,145 / 61,147  (100.0%)
  org.id                            11,731 / 61,147  ( 19.2%)


  payload.action                    15,116 / 61,147  ( 24.7%)
  payload.pull_request.number        8,338 / 61,147  ( 13.6%)
  payload.pull_request.merged            0 / 61,147  (  0.0%)


  payload.issue.number               4,863 / 61,147  (  8.0%)


  payload.commits                        0 / 61,147  (  0.0%)


## 4b. Finding: `payload.pull_request.merged` is 0% present — GitHub truncated this object

The field-presence table above shows `payload.pull_request.merged` at **0%**, which contradicts the spec's Bronze schema (Phase 2), where `pr_merged` is sourced directly from that field. This isn't a bug in this notebook — it's a real, dated change in GitHub's event feed. Checked directly below.

In [7]:
pr_events = [e for e in events if e.get("type") == "PullRequestEvent"]
key_sets = Counter()
for e in pr_events:
    pr = e.get("payload", {}).get("pull_request")
    key_sets[tuple(sorted(pr.keys())) if pr else None] += 1

print(f"PullRequestEvent count this hour: {len(pr_events):,}")
print("Distinct key-sets under payload.pull_request:")
for ks, count in key_sets.most_common():
    print(f"  count={count}:", ks)

action_counts = Counter(e["payload"].get("action") for e in pr_events)
print(f"\npayload.action distribution for PullRequestEvent: {dict(action_counts)}")


PullRequestEvent count this hour: 5,500
Distinct key-sets under payload.pull_request:
  count=5500: ('base', 'head', 'id', 'number', 'url')

payload.action distribution for PullRequestEvent: {'opened': 2176, 'merged': 1787, 'labeled': 1221, 'closed': 155, 'assigned': 82, 'unlabeled': 63, 'reopened': 16}


**What this means:** as of this measurement (checked against a 2025-06-10 sample for comparison, which *did* have the full ~44-field object including `merged`, `created_at`, `merged_at`), GitHub now sends only `base`, `head`, `id`, `number`, `url` under `payload.pull_request`. The old boolean/timestamp fields the spec's Bronze schema names are gone from the live feed.

**The good news:** `payload.action` now includes a distinct value `"merged"` (separate from `"closed"`), so "was this PR merged" is still answerable — just from the *event's own action*, not from a field inside the PR object. And because every PR lifecycle step (`opened`, `merged`, `closed`, ...) is its own timestamped event, PR lifecycle timing (open → merge duration, needed for `mart_pr_lifecycle` in Phase 5) can be derived by matching the `opened` and `merged` action events for the same `(repo_id, pr_number)` and diffing their `created_at` timestamps — an event-sourced design, not a lookup into a field that no longer exists. This changes the exact column mapping in the Bronze schema and the join logic in Phase 5's `mart_pr_lifecycle`; recorded here so it's decided deliberately in those phases, not discovered as a bug mid-build.

## 4c. Second finding: `payload.commits` is also 0% present — PushEvent lost it too

Same story, different field. The PushEvent schema tree above (Section 3) lists `payload.repository_id`, `payload.push_id`, `payload.ref`, `payload.head`, `payload.before` — no `commits` array. Checked the raw payload of an actual PushEvent directly below.

In [8]:
push_events = [e for e in events if e.get("type") == "PushEvent"]
print(f"PushEvent count this hour: {len(push_events):,}")
print("Full payload of one real PushEvent:")
print(json.dumps(push_events[0]["payload"], indent=2))


PushEvent count this hour: 40,632
Full payload of one real PushEvent:
{
  "repository_id": 1238077628,
  "push_id": 43857888439,
  "ref": "refs/heads/rag",
  "head": "446d36ab0e370745b6b758531aac439c115afad7",
  "before": "ebfa50c5e2f256c4c09bc5939ee86d25228faf1a"
}


**What this means:** unlike the PR case, there's no substitute field here — no `size` or `distinct_size` count survives either, just `before`/`head` commit SHAs. Reconstructing the actual commit count would mean calling GitHub's REST API per push (needs auth, has rate limits, defeats the point of using GH Archive's free/no-auth feed). The `commit_count` column in the spec's Bronze schema **cannot be populated from this source anymore** — worth deciding explicitly in Phase 2 whether to drop that column or keep it as always-NULL, rather than silently shipping a column that never has data.

## 5. Raw JSON vs Parquet (all columns) vs Parquet (projected columns)

This is the measurement that drives the retention decision in ADR-001. The projected columns below reflect what's *actually in the feed today* (see 4b) rather than the spec's original Bronze schema — `pr_merged`/`pr_created_at`/`pr_merged_at` are dropped and `action` is kept instead, since that's the real source of merge status now.

**Why not `read_json_auto` for the projected version too:** the section above showed `payload` varies so much across ~15 GitHub event types that DuckDB can't merge it into one struct — it falls back to a tagged union, and simple `payload.commits`-style dot access breaks with a "key not found" error depending on which shape a given row has. The fix mirrors exactly what Phase 2's `ingest.py` is designed to do: treat each line as raw JSON and pull out only the specific paths we need with explicit extraction, rather than asking an engine to auto-infer one type across wildly different shapes.

In [9]:
con = duckdb.connect()
json_path = JSON_PATH.as_posix()

# All-columns Parquet: let DuckDB infer the full, wide schema across every
# event type in this hour (sample_size=-1 => scan every row, not a sample,
# since a size measurement built on a partial schema would understate size).
all_cols_path = RAW_DIR / "sample_all_columns.parquet"
con.execute(f"""
    COPY (
        SELECT * FROM read_json_auto('{json_path}', format='newline_delimited', sample_size=-1)
    ) TO '{all_cols_path.as_posix()}' (FORMAT PARQUET, COMPRESSION ZSTD)
""")
all_cols_bytes = all_cols_path.stat().st_size

# Projected Parquet: raw JSON per line (read_json_objects does no struct inference
# at all), then explicit path extraction for only the columns we actually keep.
projected_path = RAW_DIR / "sample_projected.parquet"
con.execute(f"""
    COPY (
        SELECT
            (json->>'$.id')::BIGINT AS event_id,
            json->>'$.type' AS event_type,
            (json->>'$.created_at')::TIMESTAMP AS created_at,
            (json->>'$.actor.id')::BIGINT AS actor_id,
            json->>'$.actor.login' AS actor_login,
            (json->>'$.repo.id')::BIGINT AS repo_id,
            json->>'$.repo.name' AS repo_name,
            (json->>'$.org.id')::BIGINT AS org_id,
            json->>'$.payload.action' AS action,
            (json->>'$.payload.pull_request.number')::INT AS pr_number,
            (json->>'$.payload.issue.number')::INT AS issue_number,
            json_array_length(json->'$.payload.commits') AS commit_count
        FROM read_json_objects('{json_path}', format='newline_delimited')
    ) TO '{projected_path.as_posix()}' (FORMAT PARQUET, COMPRESSION ZSTD)
""")
projected_bytes = projected_path.stat().st_size

print(f"Raw JSON (uncompressed):     {uncompressed_bytes:>12,} bytes")
print(f"Raw JSON (gzip):             {compressed_bytes:>12,} bytes")
print(f"Parquet, all columns:        {all_cols_bytes:>12,} bytes")
print(f"Parquet, projected columns:  {projected_bytes:>12,} bytes")
print()
print(f"Compression vs raw JSON, all columns:       {uncompressed_bytes / all_cols_bytes:.1f}x")
print(f"Compression vs raw JSON, projected columns: {uncompressed_bytes / projected_bytes:.1f}x")
print(f"Projection alone shrinks it a further:      {all_cols_bytes / projected_bytes:.1f}x")


Raw JSON (uncompressed):      107,955,076 bytes
Raw JSON (gzip):               23,594,269 bytes
Parquet, all columns:          24,470,837 bytes
Parquet, projected columns:     1,703,444 bytes

Compression vs raw JSON, all columns:       4.4x
Compression vs raw JSON, projected columns: 63.4x
Projection alone shrinks it a further:      14.4x


## 6. Extrapolation: does 7-day bronze retention fit the free tier?

In [10]:
GIB = 1024 ** 3
R2_FREE_TIER_GIB = 10  # measured from Cloudflare's R2 pricing page in Phase 0

hourly_bronze_bytes = projected_bytes
daily_bronze_bytes = hourly_bronze_bytes * 24
weekly_bronze_bytes = daily_bronze_bytes * 7

# Silver (Phase 4) doesn't exist yet — it's a deduplicated, conformed copy of
# bronze, so it should be the same size or smaller. Until it's actually built
# and measured, assume it's no bigger than bronze, as a conservative upper bound.
assumed_weekly_silver_bytes = weekly_bronze_bytes
total_7day_footprint_bytes = weekly_bronze_bytes + assumed_weekly_silver_bytes

print(f"Measured bronze size for 1 hour (projected, zstd): {hourly_bronze_bytes:,} bytes")
print(f"Extrapolated bronze @ 7-day retention:  {weekly_bronze_bytes / GIB:.3f} GiB")
print(f"Assumed silver @ 7-day retention (<=bronze, not yet built): "
      f"{assumed_weekly_silver_bytes / GIB:.3f} GiB")
print(f"Combined bronze+silver footprint at steady state: "
      f"{total_7day_footprint_bytes / GIB:.3f} GiB")
print()
print(f"Cloudflare R2 free tier (Standard storage): {R2_FREE_TIER_GIB} GiB")
used_pct = 100 * (total_7day_footprint_bytes / GIB) / R2_FREE_TIER_GIB
print(f"Headroom: {R2_FREE_TIER_GIB - total_7day_footprint_bytes / GIB:.3f} GiB free "
      f"({used_pct:.1f}% of free tier used at steady state)")


Measured bronze size for 1 hour (projected, zstd): 1,703,444 bytes
Extrapolated bronze @ 7-day retention:  0.267 GiB
Assumed silver @ 7-day retention (<=bronze, not yet built): 0.267 GiB
Combined bronze+silver footprint at steady state: 0.533 GiB

Cloudflare R2 free tier (Standard storage): 10 GiB
Headroom: 9.467 GiB free (5.3% of free tier used at steady state)


## Summary

Sample hour: **2026-09-17 15:00 UTC**, an ordinary weekday hour, 61,147 events.

**Size and compression:**

| Form | Size | vs raw JSON |
|---|---|---|
| Raw JSON (uncompressed) | 107,955,076 bytes (103 MiB) | 1x (baseline) |
| Raw JSON, gzip (what GH Archive actually ships) | 23,594,269 bytes (22.5 MiB) | 4.6x smaller |
| Parquet, all columns, zstd | 24,470,837 bytes (23.3 MiB) | 4.4x smaller |
| Parquet, projected columns, zstd | 1,703,444 bytes (1.6 MiB) | **63.4x smaller** |

Projection (keeping ~12 columns instead of the full nested object) is doing almost all of the real work here — going from all-columns Parquet to projected Parquet alone is a further 14.4x reduction. Compression codec choice barely matters once you've projected; it's *which columns you keep* that dominates storage cost.

**Event type mix:** PushEvent dominates at 66% of all events; the next largest, PullRequestEvent, is under 9%. Anything that scans 'all events' should expect to mostly be scanning pushes.

**Retention verdict:** at ~1.6 MiB/hour projected, bronze at 7-day retention is ~0.27 GiB, and even the conservative bronze+silver combined estimate is ~0.53 GiB — **5.3% of Cloudflare R2's 10 GiB free tier**. The spec's 7-day default is not just adequate, it has enormous headroom — **it could safely be extended to 30+ days** without approaching the free tier limit, if there's ever a reason to (e.g. debugging a late-arriving data issue that needs more raw history). No adjustment needed for now; recorded as a real measurement, not the spec's estimate, in ADR-001.

**Two schema findings that affect Phase 2 and Phase 5** (see 4b/4c above for detail): GitHub has trimmed the live event feed since the spec was written. `payload.pull_request.merged/created_at/merged_at` and `payload.commits` no longer exist. PR merge status is still derivable (from `payload.action == "merged"` plus matching `opened`/`merged` event timestamps), but push commit counts are not recoverable from this source at all. These need explicit decisions in Phase 2 (bronze schema) and Phase 5 (`mart_pr_lifecycle`), not silent workarounds.